In [0]:
%sql
use catalog misgauravcatalog;
create database if not exists retaildb;
use retaildb; 

In [0]:
%sql
create table if not exists retaildb.orders_bronze
(
order_id int,
order_date string,
customer_id int,
order_status string,
filename string,
createdon timestamp
)
using delta
location "gs://databricksfolder/bronze/"
partitioned by (order_status)
TBLPROPERTIES(delta.enableChangeDataFeed = true);

In [0]:
%sql
create table if not exists retaildb.orders_silver
(
order_id int,
order_date date,
customer_id int,
order_status string,
order_year int GENERATED ALWAYS AS (YEAR(order_date)),
order_month int GENERATED ALWAYS AS (MONTH(order_date)),
createdon timestamp,
modifiedon timestamp
)
using delta
location "gs://databricksfolder/silver/"
partitioned by (order_status)
TBLPROPERTIES(delta.enableChangeDataFeed = true);

In [0]:
# 1. Create a 100% empty DataFrame with the exact schema matching your table
empty_silver_df = spark.sql("""
    SELECT 
      CAST(NULL AS INT) AS order_id,
      CAST(NULL AS DATE) AS order_date,
      CAST(NULL AS INT) AS customer_id,
      CAST(NULL AS STRING) AS order_status,
      CAST(NULL AS INT) AS order_year,
      CAST(NULL AS INT) AS order_month,
      CAST(NULL AS TIMESTAMP) AS createdon,
      CAST(NULL AS TIMESTAMP) AS modifiedon
    WHERE 1 = 0
""")

# 2. Write it directly to the GCS path. 
# This forces Delta Lake to physically build the _delta_log directory structure.
empty_silver_df.write \
    .format("delta") \
    .mode("append") \
    .save("gs://databricksfolder/silver/")

print("Physical storage log initialized successfully on GCS!")

In [0]:
%sql
create table if not exists retaildb.orders_gold
(
customer_id int,
order_status string,
order_year int,
num_orders int
)
using delta
location "gs://databricksfolder/gold/"
TBLPROPERTIES(delta.enableChangeDataFeed = true)

In [0]:
# 1. Create a 100% empty DataFrame with the exact schema matching your table
empty_gold_df = spark.sql("""
    SELECT 
      CAST(NULL AS INT) AS customer_id,
      CAST(NULL AS STRING) AS order_status,
      CAST(NULL AS INT) AS order_year,
      CAST(NULL AS INT) AS num_orders
    WHERE 1 = 0
""")

# 2. Write it directly to the GCS path. 
# This forces Delta Lake to physically build the _delta_log directory structure.
empty_gold_df.write \
    .format("delta") \
    .mode("append") \
    .save("gs://databricksfolder/gold/")

print("Physical storage log initialized successfully on GCS!")

In [0]:
%sql
copy into retaildb.orders_bronze from (
  select order_id::int,
  order_date::string,
  customer_id::int,
  order_status::string,
  _metadata.file_path as filename,
  current_timestamp() as createdon
  from 'gs://databricksfolder/raw/'
)
fileformat = CSV
format_options('header'='true');


In [0]:
%sql
describe history retaildb.orders_silver;

In [0]:
%sql
describe history retaildb.orders_gold;

In [0]:
# ====================================================================
# 1. Initialize & Track Watermarks Cleanly
# ====================================================================
# Find the absolute latest version available in Bronze right now
bronze_history_df = spark.sql("DESCRIBE HISTORY retaildb.orders_bronze")
current_bronze_max = bronze_history_df.select("version").first()[0]

# Create a robust tracking table if it doesn't exist
spark.sql("""
    CREATE TABLE IF NOT EXISTS retaildb.pipeline_watermarks (
        pipeline_name STRING,
        last_processed_version INT
    ) USING delta
""")

# Fetch the last processed version for our Silver pipeline
watermark_df = spark.sql("""
    SELECT last_processed_version 
    FROM retaildb.pipeline_watermarks 
    WHERE pipeline_name = 'bronze_to_silver'
""")

if watermark_df.isEmpty():
    # Brand new pipeline initialization state
    last_processed_version = -1
    # Insert initial tracking row
    spark.sql("INSERT INTO retaildb.pipeline_watermarks VALUES ('bronze_to_silver', -1)")
else:
    last_processed_version = watermark_df.first()[0]


# ====================================================================
# 2. Determine Processing State & Version Routing
# ====================================================================
# If this is the absolute first time running the pipeline (v0 and v1)
if last_processed_version == -1 and current_bronze_max <= 1:
    print(f"Processing baseline data directly from full table state (Current Version: {current_bronze_max})...")
    
    changes_df = spark.read.table("retaildb.orders_bronze") \
        .filter("order_id > 0 AND customer_id > 0 AND order_status IN ('CLOSED', 'PENDING_PAYMENT')")
    
    target_version_to_process = current_bronze_max
else:
    # Target exactly ONE version higher than what we last processed
    target_version_to_process = last_processed_version + 1
    
    # Guardrail: If no new version has landed yet, exit safely without failing
    if target_version_to_process > current_bronze_max:
        print(f"Silver is already completely up-to-date with Bronze Version {current_bronze_max}. Exiting cleanly.")
        dbutils.notebook.exit("Up to date")
        
    print(f"Targeting exactly Bronze Version {target_version_to_process} for this sequential run...")
    
    # Read Change Data Feed for ONLY this specific version directly from GCS files
    raw_cdf_df = spark.read.format("delta") \
        .option("readChangeData", "true") \
        .option("startingVersion", target_version_to_process) \
        .option("endingVersion", target_version_to_process) \
        .load("gs://databricksfolder/bronze/")
        
    # Isolate only the newest row states to eliminate duplicate rows across updates
    changes_df = raw_cdf_df \
        .filter("_change_type IN ('insert', 'update_postimage')") \
        .filter("order_id > 0 AND customer_id > 0 AND order_status IN ('CLOSED', 'PENDING_PAYMENT')")

# Expose the cleaned dataframe back to your SQL engine
changes_df.createOrReplaceTempView("orders_bronze_changes")


# ====================================================================
# 3. Run Production Upsert (MERGE) & Update Watermark
# ====================================================================
print(f"Executing Silver tier MERGE upsert logic for Version {target_version_to_process}...")

spark.sql("""
    MERGE INTO retaildb.orders_silver tgt
    USING orders_bronze_changes src ON tgt.order_id = src.order_id
    WHEN MATCHED THEN
      UPDATE SET tgt.order_status = src.order_status, 
                 tgt.customer_id = src.customer_id, 
                 tgt.modifiedon = CURRENT_TIMESTAMP()
    WHEN NOT MATCHED THEN
      INSERT (order_id, order_date, customer_id, order_status, createdon, modifiedon) 
      VALUES (src.order_id, src.order_date, src.customer_id, src.order_status, CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP())
""")

# Update our explicit watermark tracking table so the next run knows exactly where to go
spark.sql(f"""
    UPDATE retaildb.pipeline_watermarks 
    SET last_processed_version = {target_version_to_process} 
    WHERE pipeline_name = 'bronze_to_silver'
""")

print(f"Silver pipeline successfully processed and bookmarked Version {target_version_to_process}!")

In [0]:
# ====================================================================
# 1. Direct Batch Aggregation: Read current Silver table state directly
# ====================================================================
print("Calculating current up-to-date aggregates from Silver layer...")

# This reads the complete, active state of your Silver table files.
# It automatically includes all historical data plus any new file data.
changes_df = spark.sql("""
    SELECT 
        customer_id, 
        order_status, 
        order_year, 
        COUNT(order_id) AS num_orders
    FROM retaildb.orders_silver 
    GROUP BY customer_id, order_status, order_year
""")

# Expose the calculated aggregates as a temporary view for verification if needed
changes_df.createOrReplaceTempView("orders_silver_aggregates")


# ====================================================================
# 2. Run Production Overwrite (Safe for External Tables)
# ====================================================================
print("Executing Gold tier complete table overwrite...")

# By using insertInto with mode("overwrite"), Spark cleanly wipes the old data 
# inside gs://databricksfolder/gold/ and drops in the fresh totals 
# WITHOUT destroying your pre-defined DDL schema or TBLPROPERTIES.
changes_df.write \
    .mode("overwrite") \
    .insertInto("retaildb.orders_gold")

print("Gold tier pipeline processed completely and cleanly via Overwrite!")